# 02 — Silver Transform

## Story beat: "Make the data trustworthy"

Bronze has raw POS exports — mixed casing, string dates, rows with zero quantity. The **silver** layer applies **business rules**: proper types, filtered bad rows, calculated amounts, and a **dimensional model** (dimensions + fact) that analysts recognize.

We also turn on **V-Order** here — a write-time layout optimization that makes downstream Power BI and SQL reads faster and cheaper.

---

## What we build

| Silver table | Type | Source | Key transforms |
| --- | --- | --- | --- |
| `dim_store` | Dimension | `bronze.stores` | Title-case names, select core columns |
| `dim_product` | Dimension | `bronze.products` | Cast price to decimal |
| `fact_sales` | Fact | `bronze.sales` + dim | Parse timestamps, filter qty>0, compute gross/net |

---

## V-Order in one sentence

**V-Order** (Verti-Scan ordering) sorts and dictionary-encodes Parquet columns at write time so BI engines skip irrelevant row groups — up to ~50% faster targeted reads on large tables. Warehouses get it by default; in Spark you opt in.

> **Copilot callout:** Open the Copilot pane and ask *"Explain what this notebook does"* or *"Add a dedupe on sale_id before writing fact_sales"*.

In [ ]:
%run 00_config

## Step 1 — Enable V-Order for this Spark session

This session-level flag applies to **new writes**. Existing files are unchanged until you run `OPTIMIZE` (see Step 4).

In [ ]:
from pyspark.sql import functions as F

# Opt in to V-Order for this session's writes (Warehouse enables by default; Spark opts in).
spark.conf.set("spark.sql.parquet.vorder.default", "true")

## Step 2 — Build dimension tables

Dimensions describe **who/what/where** — relatively small, slowly changing. We keep them separate from the fact so reports can slice sales by region or category without repeating descriptive columns on every row.

In [ ]:
dim_store = (
    spark.table(f"{BRONZE_SCHEMA}.stores")
    .select("store_id", F.initcap("store_name").alias("store_name"), "city", "state", "state_id", "region")
)
dim_store.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{SILVER_SCHEMA}.dim_store")

dim_product = (
    spark.table(f"{BRONZE_SCHEMA}.products")
    .select("product_id", "product_name", "category", F.col("unit_price").cast("decimal(10,2)").alias("unit_price"))
)
dim_product.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{SILVER_SCHEMA}.dim_product")
print("dimensions written: silver.dim_store, silver.dim_product")

## Step 3 — Build the sales fact table

The fact table holds **measurable events** (sales). Logic:
1. Parse `sale_ts` to a real timestamp; derive `sale_date` for daily rollups.
2. **Filter** `quantity > 0` — returns and voids excluded at silver (business rule).
3. **Join** product dimension for `unit_price`.
4. **Calculate** `gross_amount = qty × price` and `net_amount` after discount.

> **Presenter note:** Walk through one row verbally — *"Store 12 sold 3 units of product 7 at 10% off → here's gross vs net."*

In [ ]:
fact = (
    spark.table(f"{BRONZE_SCHEMA}.sales")
    .withColumn("sale_ts", F.to_timestamp("sale_ts"))
    .withColumn("sale_date", F.to_date("sale_ts"))
    .filter(F.col("quantity") > 0)
    .join(dim_product.select("product_id", "unit_price"), "product_id", "left")
    .withColumn("gross_amount", F.col("quantity") * F.col("unit_price"))
    .withColumn("net_amount", F.round(F.col("gross_amount") * (1 - F.col("discount_pct")/100.0), 2))
    .select("sale_id", "sale_ts", "sale_date", "store_id", "product_id", "quantity", "discount_pct", "gross_amount", "net_amount")
)
fact.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{SILVER_SCHEMA}.fact_sales")
print(f"{SILVER_SCHEMA}.fact_sales: {fact.count()} rows")

## Step 4 — Persist V-Order and compact the fact table

Two operations:
1. **`ALTER TABLE ... SET TBLPROPERTIES`** — marks the table as V-Order enabled for future writes.
2. **`OPTIMIZE`** — rewrites existing small files into fewer, V-Ordered files (also handles small-file problem).

Module 4 (Direct Lake) and Module 2 (warehouse reads) benefit from this layout.

In [ ]:
spark.sql(f"ALTER TABLE {SILVER_SCHEMA}.fact_sales SET TBLPROPERTIES ('delta.parquet.vorder.enabled' = 'true')")
spark.sql(f"OPTIMIZE {SILVER_SCHEMA}.fact_sales")
display(spark.sql(f"SELECT * FROM {SILVER_SCHEMA}.fact_sales LIMIT 10"))

---

## ✅ Success looks like

- `silver.dim_store`, `silver.dim_product`, `silver.fact_sales` visible in the lakehouse.
- Sample fact rows show parsed dates and net amounts.

**Next notebook:** `03_gold_aggregate` — roll up to business KPIs for Power BI.